The only important variables to set in this entire script are found in the second cell, and are named:

1. **subject_ID**
2. **user**
3. **execute**

The directory and paths are all set to *Omri's* directory. It doesn't make sense to store the data in two places on Milgram since *Aryan* already has access to data in *Omri's* scratch folder. Thus, these don't need to be changed or manipulated.

Also, the following link is the BIDS specification [https://bids-specification.readthedocs.io/en/stable/modality-specific-files/task-events.html]

It contains the relevant information to specify renaming formats. I am sure of the following:

1. Functional Runs: sub-multimem-p001_task-multisensorymemory_run-1_bold.nii.gz
2. Anatomical Scans: sub-multimem-p001_anat-T1w.nii.gz
3. Events Files: sub-multimem-p001_task-multisensorymemory_run-1_events.csv

Our events files should actually be present in two formats, **.tsv** *(not a typo)* and **.json** (explained in the link).

In [3]:
import csv
import os
import json
import shutil
import pandas as pd

# Set the subject we are renaming as a 3 digit string, 
# Set the user's NET ID
# Only set execute to true if we want to perform the renaming, otherwise only prints calculated names
subject_ID = 'mm01'
user = "or62"
execute = False

######################## No other variables to be hard-coded ########################################################################

# Specify directory with nii files
directory = f'/gpfs/milgram/scratch60/turk-browne/{user}/sandbox/{subject_ID}_nii/'
bids_dir = f'/gpfs/milgram/scratch60/turk-browne/{user}/sandbox/{subject_ID}_nii_bids/'

# Choose all non '.' files in the directory
files = [f for f in os.listdir(directory) if f.endswith('.json') or f.endswith('.nii') or f.endswith('.csv') or f.endswith('.mat')]
files = [f for f in files if 'practice' not in f and 'order' not in f]

# Sort so that we rename nii files first (allows JSON reference with same name)
files = sorted(files, key=lambda x: (x.endswith('.json'), x))

for file in files:
    print(file)

events_run_0_sub_mm01.mat
events_run_1_sub_mm01.mat
events_run_2_sub_mm01.mat
events_run_3_sub_mm01.mat
events_run_4_sub_mm01.mat
events_run_5_sub_mm01.mat
events_run_6_sub_mm01.mat
events_run_7_sub_mm01.mat
events_run_8_sub_mm01.mat
events_run_9_sub_mm01.mat
mm01_20241016145206_1.nii
mm01_20241016145206_10.nii
mm01_20241016145206_11.nii
mm01_20241016145206_12.nii
mm01_20241016145206_13.nii
mm01_20241016145206_14.nii
mm01_20241016145206_15.nii
mm01_20241016145206_16.nii
mm01_20241016145206_17.nii
mm01_20241016145206_18.nii
mm01_20241016145206_19.nii
mm01_20241016145206_2.nii
mm01_20241016145206_3.nii
mm01_20241016145206_4.nii
mm01_20241016145206_5.nii
mm01_20241016145206_6.nii
mm01_20241016145206_7.nii
mm01_20241016145206_8.nii
mm01_20241016145206_9.nii
mm01_20241016145206_1.json
mm01_20241016145206_10.json
mm01_20241016145206_11.json
mm01_20241016145206_12.json
mm01_20241016145206_13.json
mm01_20241016145206_14.json
mm01_20241016145206_15.json
mm01_20241016145206_16.json
mm01_20241016

In [4]:
# Helper function to locate the JSON file corresponding to a CSV events file
def find_json(directory, files, run):
    for file in files:
        if file.endswith('.json'):
            with open(os.path.join(directory, file), 'r') as json_file:
                json_data = json.load(json_file)
                if f"run-{run}" in json_data['SeriesDescription']:
                    return file    
    return 0

In [6]:
# Helper function to tranform .mat files to CSV files
def transform_mat_to_csv(path_to_events_file, directory):
    # read data from nested .mat structure
    events = loadmat(file_name=path_to_events_file)
    onset = events['data']['Onsets'][0][0]
    duration = events['data']['Durations'][0][0]
    stimulus_name = events['data']['trialType'][0][0]
    
    # convert values to arrays
    onset = np.concatenate(onset).ravel()
    duration = np.concatenate(duration).ravel()
    stimulus_name = np.concatenate(stimulus_name).ravel()
    
    # create panda with columns onset duration and stimulus name
    events_panda = pd.DataFrame({'onset': onset, 'duration': duration, 'trial_type': stimulus_name})
    
    events_panda.to_csv(directory + "")
    
    return 0

In [10]:
# loop through events files and save as csv
mat_files = [f for f in os.listdir(directory) if f.endswith('.mat')]
print(mat_files[0])


events_run_7_sub_mm01.mat


In [3]:
# Helper function to transform CSV events files to TSV
def transform_csv_to_tsv(path):
    df = pd.read_csv(path)
    new_path = path.replace(".csv", ".tsv")
    df.to_csv(new_path, sep='\t', index=False)
    os.remove(path)
    
    return 0

In [7]:
# Example Usage: find the JSON file corresponding to run 1 inside files
find_json(directory, files, "1")

'mm01_20241016145206_7.json'

In [5]:
# This function performs the main renaming of the files when given arguments
# called files and directory, which specify the files to be renamed
def rename(directory, files):
    names = []
    # loop over every file to be renamed 
    for filename in files:
        file_path = os.path.join(directory, filename)
        
        # This conditional sets the path to reference the .json where the "code" is extracted form
        # The code specifies the type of scan and for nii files we reference the correspondingly named json for extraction
        
        if filename.endswith('.json'):
            json_path = file_path
        elif filename.endswith('.nii'):
            json_path = os.path.join(directory, f"{filename.split('.')[0]}.json")
        elif filename.endswith('.csv'):
            json_path = os.path.join(directory, find_json(directory, files, filename.split('_')[2].split('.')[0][1]))
        
        with open(json_path, 'r') as json_file:
            json_data = json.load(json_file)
            code = json_data['SeriesDescription']
            # This conditional adds the acquisition time to audio test file names so if we repeat then we don't over write files
            if 'audiotest' in code:
                acq_time = f"_{json_data['AcquisitionTime']}"
            else:
                acq_time = ""
            
        # These conditionals allow for specific formatting (according to BIDS) for the various scans
        if 'anat' in code:
            # This is an anatomical run
            new_path = f"sub-{filename[:13].replace('_p', '')}_{code}.{filename.split('.')[-1]}"
        elif 'Scout' in code:
            # This is a scout run
            new_path = f"sub-{filename[:13].replace('_p', '')}_{code}_scout.{filename.split('.')[-1]}"
        elif 'epi' in code:
            # This is a fieldmap
            new_path = f"sub-{filename[:13].replace('_p', '')}_{code}.{filename.split('.')[-1]}"
        else:
            # This is a functional run or the .csv file for the run (including audio test)
            if filename.endswith('.nii') or filename.endswith('.json'):
                new_path = f"sub-{filename[:13].replace('_p', '')}_{code}_bold{acq_time}.{filename.split('.')[-1]}"
            elif filename.endswith('.csv'):
                new_path = f"sub-{json_path.split('/')[-1][:13].replace('_p', '')}_{code}_events{acq_time}.{filename.split('.')[-1]}"

        # Conditional to rename/print.
        if execute:
            os.rename(file_path, os.path.join(directory, new_path))
            if new_path.endswith(".csv"): transform_csv(os.path.join(directory, new_path))
        else:
            print(new_path)
        names.append(new_path)
    print("Names have been assigned")

    return names

In [8]:
# Usage with execute set as False
new_names = rename(directory, files)

Names have been assigned


In [13]:
# This function moves all the files to the BIDS directory
# Any files not in bids go to /bids/sourcedata/sub-[]/other/
def move(niidir, datadir):
    subject_dir = datadir + subject_id
    if execute: os.chdir(datadir)
    
    maps = {'scout': 'other', 'practice': 'other', 'order': 'other', 'audiotest': 'other', 'anat': 'anat', 'run': 'func', 'epi': 'fmap'}
    
    for file in os.listdir(niidir):
        for val in maps.keys():
            if val in file:
                if execute: 
                    if not os.path.exists(f"{subject_dir}/{maps[val]}"): os.makedirs(f"{subject_dir}/{maps[val]}")
                    shutil.move(os.path.join(niidir, file), f"{subject_dir}/{maps[val]}")
                print(f"{file} sent to {maps[val]}")
                break
    if execute:
        os.chdir(subject_dir)
        shutil.move(f"{subject_dir}/other", f"{datadir}/sourcedata/{subject_id}/other")
    else:
        print(f"Would have moved {subject_dir}/other to {datadir}/sourcedata/{subject_id}/other")
    return 1


In [14]:
move(directory, bids_dir)

sub-multimem002_task-multisensorymemory_run-2_bold.nii sent to func
sub-multimem002_anat-T2w_acq-hipp.json sent to anat
sub-multimem002_dir-AP_epi.json sent to fmap
sub-multimem002_AAHead_Scout_64ch-head-coil_scout.nii sent to other
sub-multimem002_task-multisensorymemory_run-7_bold.nii sent to func
sub-multimem002_task-multisensorymemory_run-3_bold.json sent to func
sub-multimem002_task-multisensorymemory_run-2_bold.json sent to func
sub-multimem002_task-multisensorymemory_run-1_bold.json sent to func
sub-multimem002_AAHead_Scout_64ch-head-coil_MPR_sag_scout.json sent to other
sub-multimem002_task-multisensorymemory_run-6_bold.nii sent to func
sub-multimem002_task-multisensorymemory_run-9_bold.json sent to func
sub-multimem002_AAHead_Scout_64ch-head-coil_MPR_cor_scout.nii sent to other
run_order_P002.npy sent to other
sub-multimem002_dir-AP_epi.nii sent to fmap
sub-multimem002_AAHead_Scout_64ch-head-coil_MPR_cor_scout.json sent to other
sub-multimem002_task-multisensorymemory_run-4_ev

1

In [46]:
# This function formats JSON files for fieldmap files, functional files, and anatomical files
# Functional: Adds 'TaskName', deletes 'AcquisitionDuration'
# Anatomical: Deletes 'RepetitionTime'
# Fieldmaps: Adds 'IntendedFor', deletes 'RepetitionTime'
def format_json(bids_dir):
    path_func = bids_dir + subject_ID + "/func"
    path_anat = bids_dir + subject_ID + "/anat"
    path_fmap = bids_dir + subject_ID + "/fmap"
    fmap_1 = bids_dir + subject_ID + "/fmap/" + subject_ID + "_dir-AP_epi.json"
    fmap_2 = bids_dir + subject_ID + "/fmap/" + subject_ID + "_dir-PA_epi.json"
    
    for file in os.listdir(path_func):
        if file.endswith(".nii"):
            # Edit the first fieldmap JSON. Add IntendedFor and delete repetition time
            with open(fmap_1, 'r') as json_file_1:
                json_data_1 = json.load(json_file_1)
            
            if 'IntendedFor' not in json_data_1: json_data_1['IntendedFor'] = []
            json_data_1['IntendedFor'].append(f"bids::{subject_ID}/func/{file}")
            
            if 'RepetitionTime' in json_data_1: del json_data_1['RepetitionTime']
            
            with open(fmap_1, 'w') as json_file_1:
                json.dump(json_data_1, json_file_1)
            
            # Edit the second fieldmap JSON. Add IntendedFor and delete repetition time
            with open(fmap_2, 'r') as json_file_2:
                json_data_2 = json.load(json_file_2)
            
            if 'IntendedFor' not in json_data_2: json_data_2['IntendedFor'] = []
            json_data_2['IntendedFor'].append(f"bids::{subject_ID}/func/{file}")
            
            if 'RepetitionTime' in json_data_2: del json_data_2['RepetitionTime']
            
            with open(fmap_2, 'w') as json_file_2:
                json.dump(json_data_2, json_file_2)
            
            print(f"Fieldmap jsons updated for {file}")
        elif file.endswith(".json"):
            # Edit the function JSON files. Add in TaskName and remove AcquisitionDuration
            with open(os.path.join(path_func, file), 'r') as func_json:
                func_data = json.load(func_json)
            
            func_data['TaskName'] = 'multisensorymemory'
            if 'AcquisitionDuration' in func_data: del func_data['AcquisitionDuration']
            
            with open(os.path.join(path_func, file), 'w') as func_json:
                json.dump(func_data, func_json)
            
            print(f"Fieldmap jsons updated for {file}")

    for file in os.listdir(path_anat):
        # Edit the anatomical data JSON files. Remove repetition time. 
        if file.endswith(".json"):
            with open(os.path.join(path_anat, file), 'r') as anat_json:
                anat_data = json.load(anat_json)
            if 'RepetitionTime' in anat_data: del anat_data['RepetitionTime']
            
            with open(os.path.join(path_anat, file), 'w') as anat_json:
                json.dump(anat_data, anat_json)
        
            print(f"Anatomical JSON edited for {file}")
                
    print("Formatting of JSONs is done")
    
    return 0


In [22]:
if execute: format_json(bids_dir)

Fieldmap jsons updated for sub-multimem002_task-multisensorymemory_run-2_bold.nii
Fieldmap jsons updated for sub-multimem002_task-multisensorymemory_run-7_bold.nii
Fieldmap jsons updated for sub-multimem002_task-multisensorymemory_run-3_bold.json
Fieldmap jsons updated for sub-multimem002_task-multisensorymemory_run-2_bold.json
Fieldmap jsons updated for sub-multimem002_task-multisensorymemory_run-1_bold.json
Fieldmap jsons updated for sub-multimem002_task-multisensorymemory_run-6_bold.nii
Fieldmap jsons updated for sub-multimem002_task-multisensorymemory_run-9_bold.json
Fieldmap jsons updated for sub-multimem002_task-multisensorymemory_run-4_events.tsv
Fieldmap jsons updated for sub-multimem002_task-multisensorymemory_run-4_bold.nii
Fieldmap jsons updated for sub-multimem002_task-multisensorymemory_run-6_bold.json
Fieldmap jsons updated for sub-multimem002_task-multisensorymemory_run-5_bold.json
Fieldmap jsons updated for sub-multimem002_task-multisensorymemory_run-8_bold.nii
Fieldmap

In [1]:
# Should be populated as soon as possible, so that we don't have to rerun everything later on
# levels is the categories variables can take
levels_dict = {'stimulus_name': {'category 1': 'some explanation', 'category 2': 'some explanation'}, 
               'event': {'category 1': 'some explanation', 'category 2': 'some explanation'}
              }

descriptions = {'onset': 'something', 'duration': 'something', 'tr': 'something', 'stimulus_name': 'something', 'event': 'something'}

In [30]:
"""
The following function completes two purposes. 
First, it edits the .tsv file for each functional run and puts them in BIDS format. 
Second, it creates the required .json files for each .tsv events file which describe the column headers. 

Inputs:
bids_dir: Path to the BIDS dataset
descriptions: The description field as a dictionary for the .json file
levels_dict: The levels field as a dictionary for the .json file

Output:
If execute = True, it will update the TSV and JSON event files
Otherwise, it will print out path to JSON files, the data to be sent to them, and one example events.tsv file's updated data
"""

def format_events(bids_dir, descriptions, levels_dict):
    path = bids_dir + subject_ID + "/func"
    
    for file in os.listdir(path):
        if file.endswith(".tsv"):
            # Fixing column headers for the events file
            temp = pd.read_csv(os.path.join(path, file), sep='\t')
            temp = temp.drop(columns=['Unnamed: 0'])
            temp = temp.rename(columns={'Time':'onset', 'TR':'tr', 'StimulusName':'stimulus_name', 'Event': 'event'})
            temp["duration"] = temp['onset'].shift(-1) - temp['onset']
            temp = temp[['onset', 'duration', 'tr', 'stimulus_name', 'event']]
            temp.at[temp.index[-1], 'duration'] = 0
            
            # Creating the required JSON files
            data_dict = {}
            for column in temp.columns:
                data_dict[column] = {"Description": f"{descriptions[column]}"}
                if column in levels_dict.keys(): data_dict[column]["Levels"] = levels_dict[column]
            
            json_filename = os.path.splitext(os.path.join(path, file))[0] + ".json"
            
            if execute:
                # Write changes to the .tsv file
                temp.to_csv(os.path.join(path, file), sep='\t', index=False)
                
                # Write a new .json file
                with open(json_filename, 'w') as f:
                    json.dump(data_dict, f)
            else:
                print(json_filename)
                print(data_dict)
    print(temp)        
    return 0


In [31]:
format_events(bids_dir, descriptions, levels_dict)

            onset  duration   tr stimulus_name                         event
0        0.027800  0.009415    0          None  TRCountdown and instructions
1        0.037215  0.008438    0          None  TRCountdown and instructions
2        0.045653  0.016891    0          None  TRCountdown and instructions
3        0.062544  0.016712    0          None  TRCountdown and instructions
4        0.079256  0.016523    0          None  TRCountdown and instructions
...           ...       ...  ...           ...                           ...
26484  502.679962  0.016726  326          None      TRCountdown and Feedback
26485  502.696688  0.016692  326          None      TRCountdown and Feedback
26486  502.713380  0.016689  326          None      TRCountdown and Feedback
26487  502.730069  0.001601  326          None      TRCountdown and Feedback
26488  502.731670  0.000000  327    End of Run                 Run completed

[26489 rows x 5 columns]


0

In [51]:
# Quick function to fix the slight problem in anatomical scan names
def fix_anat():
    path = bids_dir + subject_ID + "/anat"
    
    for file in os.listdir(path):
        if 'T2w' in file:
            new_file = f"{file.split('_')[0]}_{file.split('_')[1]}.{file.split('.')[-1]}"
            new_file = new_file.replace("anat-", "")
            if execute: os.rename(os.path.join(path, file), os.path.join(path, new_file))
            print(new_file)
        elif 'T1w' in file:
            new_file = file
            new_file = new_file.replace("anat-", "")
            if execute: os.rename(os.path.join(path, file), os.path.join(path, new_file))
            print(new_file)
         
    return 0


In [52]:
fix_anat()

sub-multimem002_T2w.json
sub-multimem002_T1w.nii
sub-multimem002_T1w.json
sub-multimem002_T2w.nii


0

Things to do:

1. ~Fix the column names of the events file~
2. ~Create code for accompanying JSON file generation of events file~
3. ~Execute events code for 002 subject data (which is already in BIDS)~
4. ~Figure out participants tsv file - NOT REQUIRED~
5. ~Figure out the dataset description file - REQUIRED~
6. ~Also needed: readme (required), citation (recommended), license (recommended), change log (optional)~
7. ~Comment out previous code~

~Change sub- as the folder name DONE~

~Change multimem-p002 to multimem002 DONE~

~Add .bidsignore for the other files (fixing by moving other to sourcedata) DONE~

~Change NaN value in events duration to 0's DONE~

~Rename anat folder files to remove anat in filenames DONE~

~Add task name to the json files for functional data DONE~

~remove acquisition duration for functional and repetition time for anatomical and fmap -> DONE~

~Fix the dataset_description field (upload new to milgram) DONE~